# 02 — Canny-guided detail Offline Preparation

Export this version's policy once from original sources into a dated run: S20 train, calibration val (checkpoint selection) and development test (reporting), with shared five-class labels and JPEG95.

**Inputs:** v4.yaml (this folder), the shared split table and original non-final sources.

**Outputs:** runs/v4-YYYY-MM-DD/{config.yaml,data/{images,labels,data.yaml,metadata.csv}} only when ALLOW_PREPARATION=True.

**Prerequisites:** Saved config/table and the BDD sources. Never runs notebook 00 or tunes a policy.

**Outline**

- [1. Configuration](#section-1)
- [2. Offline Export](#section-2)
- [3. Preparation Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

RUN_DATE=None creates today's run when preparing, otherwise inspects the newest run. Saved paths are project-relative.

In [ ]:
import os

import pandas as pd
from IPython.display import display

from vehicle_detection.common import project_root, require_paths
from vehicle_detection.experiment_data import load_experiment, prepare_data, verify_data

ROOT = project_root()
os.chdir(ROOT)  # Run All behaves the same from any kernel working directory.
EXPERIMENT = "v4"
require_paths(
    config=ROOT / f"notebooks/experiments/{EXPERIMENT}/{EXPERIMENT}.yaml",
    shared=ROOT / "configs/data.yaml",
)
config = load_experiment(ROOT, EXPERIMENT)

RUN_DATE = None
ALLOW_PREPARATION = False

<a id="section-2"></a>

## 2. Offline Export

IDs/GT/geometry are identical for every version; the policy runs on original pixels before encoding, never in the dataloader.

In [ ]:
if not ALLOW_PREPARATION:
    print(
        "Data preparation disabled; inspecting saved artifacts only.",
        "Set ALLOW_PREPARATION=True to opt in.",
    )
prepared = None
if ALLOW_PREPARATION:
    prepared = prepare_data(ROOT, EXPERIMENT, RUN_DATE)
else:
    try:
        prepared = verify_data(ROOT, EXPERIMENT, RUN_DATE)
    except FileNotFoundError as error:
        print(error)
if prepared:
    display(
        pd.Series({k: prepared[k] for k in ("run", "path", "training_subset", "policy_sha256")})
    )

<a id="section-3"></a>

## 3. Preparation Summary

An existing run whose policy changed fails; prepare a new run date instead.

In [ ]:
if prepared:
    metadata = pd.read_csv(ROOT / prepared["path"] / "metadata.csv", keep_default_na=False)
    display(
        metadata.groupby("split").agg(
            images=("id", "size"),
            applied=("applied", lambda s: (s != "").mean()),
            brightness=("brightness", "mean"),
        )
    )
print("Final-test pixels remain sealed.")